# Notebook 07: ROI Event, Journey & Scoring

Pipeline xử lý sự kiện ROI, trạng thái hành trình, và tính điểm nghi vấn.

---

## Các trạng thái hành trình (Journey States)

| Trạng thái | Mô tả |
|---|---|
| `observed` | Chưa có dữ liệu đủ để đánh giá |
| `ticket_evidence_found` | Đã có bằng chứng vé rõ ràng |
| `waiting_for_ticket_evidence` | Trong khoảng thời gian chờ cho phép |
| `uncertain_insufficient_evidence` | **MỚI**: Re-ID confidence thấp, không đủ bằng chứng |
| `suspected_no_ticket_evidence` | Nghi vấn cao: vượt thời gian + không có bằng chứng vé |

> ⚠️ **Quan trọng**: `suspected_no_ticket_evidence` KHÔNG đồng nghĩa với vi phạm đã xác nhận.
> Đây là tín hiệu ưu tiên để nhân viên xem xét thủ công.

In [1]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import pandas as pd
import numpy as np

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / 'src').exists() and (p / 'configs').exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv
from src.journey_state import build_journey_states
from src.scoring import compute_suspicion_score, alert_level_from_score, alert_level_from_journey_state, build_alert_reason

scoring_cfg = load_yaml(PROJECT_ROOT / 'configs/scoring_config.yaml')
weights = scoring_cfg['weights']

global_df = pd.read_csv(PROJECT_ROOT / 'outputs/global_ids/global_ids.csv')

global_df.head()

,global_id,entry_camera,entry_tracklet_id,ticket_camera,ticket_tracklet_id,ticket_evidence,reid_score,entry_time_sec
0,GID_00000,cam_entrance_01,1,cam_ticket_01,1.0,True,0.913689,0.0
1,GID_00001,cam_entrance_01,2,NaN,NaN,False,NaN,0.0
2,GID_00002,cam_entrance_01,3,NaN,NaN,False,NaN,0.0
3,GID_00003,cam_entrance_01,4,cam_ticket_01,4.0,True,0.917324,0.0
4,GID_00004,cam_entrance_01,5,cam_ticket_01,5.0,True,0.685172,0.0


# Tạo event mô phỏng nghiệp vụ (Realistic Simulation)

Nâng cấp từ uniform random sang:
- **Poisson arrival** cho thời gian vào cổng (realistic visitor flow)
- **Normal distribution** cho thời gian mua vé (μ=180s, σ=60s)
- **Nhóm khách** (group arrivals) mô phỏng gia đình/tour group
- **Dwell time log-normal** thực tế hơn

In [2]:
# Tạo events với phân bố thực tế hơn cho bài nghiên cứu.
np.random.seed(42)

n_people = len(global_df)

# --- Poisson arrival process ---
# Inter-arrival time ~ Exponential(mean=20s) → ~3 người/phút
inter_arrival = np.random.exponential(scale=20.0, size=n_people)
arrival_times = np.cumsum(inter_arrival)

# Group arrivals: 30% người đến theo nhóm 2-4 (cùng thời điểm ± 3s)
group_prob = 0.30
for i in range(1, n_people):
    if np.random.rand() < group_prob:
        arrival_times[i] = arrival_times[i-1] + np.random.uniform(0, 3.0)

# Sắp xếp lại
arrival_times = np.sort(arrival_times)

event_rows = []

for idx, (_, row) in enumerate(global_df.iterrows()):
    gid = row['global_id']
    entry_time = float(arrival_times[idx])

    # Event 1: entered_area
    event_rows.append({
        'global_id': gid,
        'camera_id': 'cam_entrance_01',
        'local_track_id': int(row['entry_tracklet_id']),
        'event_type': 'entered_area',
        'timestamp_sec': entry_time,
        'frame_id': None,
        'roi_name': 'entrance_line',
        'dwell_time_sec': 0.0
    })

    # Event 2: ticket_counter_evidence (nếu có vé)
    if row['ticket_evidence'] == True:
        # Thời gian di chuyển đến quầy vé: Normal(μ=180s, σ=60s), clip [30, 600]
        travel_time = np.clip(np.random.normal(180, 60), 30, 600)
        ticket_time = entry_time + travel_time
        # Dwell time tại quầy: Log-normal (median ~20s)
        dwell = float(np.random.lognormal(mean=3.0, sigma=0.5))
        dwell = np.clip(dwell, 5, 120)

        event_rows.append({
            'global_id': gid,
            'camera_id': 'cam_ticket_01',
            'local_track_id': row['ticket_tracklet_id'],
            'event_type': 'ticket_counter_evidence',
            'timestamp_sec': float(ticket_time),
            'frame_id': None,
            'roi_name': 'ticket_counter_zone',
            'dwell_time_sec': float(dwell)
        })

events_df = pd.DataFrame(event_rows)

out_path = PROJECT_ROOT / 'outputs/events/events.csv'
save_csv(events_df, out_path)

print('Saved:', out_path)
print(f'Total events: {len(events_df)}')
print(f'Entry events: {(events_df["event_type"]=="entered_area").sum()}')
print(f'Ticket events: {(events_df["event_type"]=="ticket_counter_evidence").sum()}')
print(f'Arrival time range: {arrival_times[0]:.1f}s ~ {arrival_times[-1]:.1f}s')
events_df.head()

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\events\events.csv
Total events: 583
Entry events: 364
Ticket events: 219
Arrival time range: 9.4s ~ 6912.7s


,global_id,camera_id,local_track_id,event_type,timestamp_sec,frame_id,roi_name,dwell_time_sec
0,GID_00000,cam_entrance_01,1.0,entered_area,9.385362,None,entrance_line,0.000000
1,GID_00000,cam_ticket_01,1.0,ticket_counter_evidence,99.219461,None,ticket_counter_zone,17.630536
2,GID_00001,cam_entrance_01,2.0,entered_area,69.587790,None,entrance_line,0.000000
3,GID_00002,cam_entrance_01,3.0,entered_area,95.922704,None,entrance_line,0.000000
4,GID_00003,cam_entrance_01,4.0,entered_area,114.181555,None,entrance_line,0.000000


# Journey state management

In [3]:
# Dựa trên dữ liệu sự kiện và global_id, xác định trạng thái hành trình và tính điểm cảnh báo cho từng người.
current_time_sec = scoring_cfg['scoring']['current_time_sec']
max_travel_time_sec = scoring_cfg['scoring']['max_travel_time_sec']

journey_df = build_journey_states(
    global_df=global_df,
    events_df=events_df,
    current_time_sec=current_time_sec,
    max_travel_time_sec=max_travel_time_sec
)

out_path = PROJECT_ROOT / 'outputs/journeys/journey_states.csv'
save_csv(journey_df, out_path)

print('Saved:', out_path)
print(journey_df['journey_state'].value_counts().to_string())
journey_df.head()

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\journeys\journey_states.csv
journey_state
ticket_evidence_found           219
waiting_for_ticket_evidence     137
suspected_no_ticket_evidence      8


,global_id,has_entry,has_ticket_evidence,entry_time_sec,time_since_entry_sec,time_exceeded,reid_confidence,journey_state
0,GID_00000,True,True,9.385362,990.614638,True,0.913689,ticket_evidence_found
1,GID_00001,True,False,69.587790,930.412210,True,NaN,suspected_no_ticket_evidence
2,GID_00002,True,False,95.922704,904.077296,True,NaN,suspected_no_ticket_evidence
3,GID_00003,True,True,114.181555,885.818445,True,0.917324,ticket_evidence_found
4,GID_00004,True,True,117.574053,882.425947,True,0.685172,ticket_evidence_found


# Suspicion scoring + alert generation

In [4]:
# Dựa trên dữ liệu sự kiện và global_id, xác định trạng thái hành trình và tính điểm cảnh báo cho từng người.
alert_threshold = scoring_cfg['scoring']['alert_threshold']
watch_threshold = scoring_cfg['scoring']['watch_threshold']

alert_rows = []

for _, row in journey_df.iterrows():
    gid = row['global_id']

    gid_info = global_df[global_df['global_id'] == gid].iloc[0]

    has_entry = bool(row['has_entry'])
    has_ticket = bool(row['has_ticket_evidence'])
    time_exceeded = bool(row['time_exceeded'])
    journey_state = row['journey_state']

    # Lấy reid_confidence — dùng cho uncertainty calibration
    reid_conf_raw = gid_info['reid_score']
    assoc_conf = float(reid_conf_raw) if pd.notna(reid_conf_raw) else 0.8
    reid_confidence = float(reid_conf_raw) if pd.notna(reid_conf_raw) else None

    # Tính suspicion score với uncertainty calibration
    score = compute_suspicion_score(
        has_entry=has_entry,
        has_ticket_evidence=has_ticket,
        time_exceeded=time_exceeded,
        track_quality=0.9,
        assoc_confidence=assoc_conf,
        reid_confidence=reid_confidence,  # NEW: uncertainty calibration
        weights=weights
    )

    # Xác định alert level dựa trên cả journey_state và score
    # uncertain_insufficient_evidence luôn -> LOW (không tạo cảnh báo)
    alert_level, status = alert_level_from_journey_state(
        journey_state=journey_state,
        score=score,
        alert_threshold=alert_threshold,
        watch_threshold=watch_threshold
    )

    reason = build_alert_reason(
        has_entry=has_entry,
        has_ticket_evidence=has_ticket,
        time_exceeded=time_exceeded,
        journey_state=journey_state  # NEW: xử lý uncertain case
    )

    alert_rows.append({
        'global_id': gid,
        'entry_time_sec': float(row['entry_time_sec']),
        'time_since_entry_sec': float(row['time_since_entry_sec']),
        'ticket_evidence': bool(has_ticket),
        'journey_state': journey_state,
        'reid_confidence': reid_confidence,
        'suspicion_score': float(score),
        'alert_level': alert_level,
        'status': status,
        'reason': reason
    })

alerts_df = pd.DataFrame(alert_rows)

out_path = PROJECT_ROOT / 'outputs/alerts/alerts.csv'
save_csv(alerts_df, out_path)

print('Saved:', out_path)
print('\nAlert Status:')
print(alerts_df['status'].value_counts().to_string())
print('\nJourney State Distribution:')
print(alerts_df['journey_state'].value_counts().to_string())
alerts_df.head()

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\alerts\alerts.csv

Alert Status:


status
no_alert           204
alert_generated    145
watchlist           15

Journey State Distribution:
journey_state
ticket_evidence_found           219
waiting_for_ticket_evidence     137
suspected_no_ticket_evidence      8


,global_id,entry_time_sec,time_since_entry_sec,ticket_evidence,journey_state,reid_confidence,suspicion_score,alert_level,status,reason
0,GID_00000,9.385362,990.614638,True,ticket_evidence_found,0.913689,0.677053,medium,watchlist,Entered area detected; Exceeded maximum travel...
1,GID_00001,69.587790,930.412210,False,suspected_no_ticket_evidence,NaN,0.960000,high,alert_generated,Entered area detected; No ticket counter evide...
2,GID_00002,95.922704,904.077296,False,suspected_no_ticket_evidence,NaN,0.960000,high,alert_generated,Entered area detected; No ticket counter evide...
3,GID_00003,114.181555,885.818445,True,ticket_evidence_found,0.917324,0.677599,medium,watchlist,Entered area detected; Exceeded maximum travel...
4,GID_00004,117.574053,882.425947,True,ticket_evidence_found,0.685172,0.642776,medium,watchlist,Entered area detected; Exceeded maximum travel...


# Thống kê alert

In [5]:
# Thống kê phân bố alert cuối cùng.
print('Alert Status:')
print(alerts_df['status'].value_counts().to_string())
print()
print('Alert Level x Status:')
print(alerts_df.groupby(['alert_level', 'status']).size().reset_index(name='count').to_string(index=False))

Alert Status:
status
no_alert           204
alert_generated    145
watchlist           15

Alert Level x Status:
alert_level          status  count
       high alert_generated    145
        low        no_alert    204
     medium       watchlist     15
